In [1]:
import pandas as pd
import random

random.seed(42)

categories = {
    "Payroll": {
        "Salary": ["Late Disbursement", "Incorrect Amount"],
        "Bonus": ["Missing Bonus", "Calculation Error"],
    },
    "Leave": {
        "Annual Leave": ["Balance Discrepancy", "Approval Delay"],
        "Sick Leave": ["Documentation Issue", "Balance Discrepancy"],
    },
    "Benefits": {
        "Insurance": ["Enrollment Issue", "Claim Rejection"],
        "Retirement": ["Contribution Error", "Enrollment Issue"],
    },
    "Workplace": {
        "Harassment": ["Verbal", "Discrimination"],
        "Facilities": ["Equipment Request", "Seating Issue"],
    },
}

intents = ["Complaint", "Query", "Request", "Feedback"]
sentiments = ["Positive", "Negative", "Neutral"]

feedback_examples = {
    "Late Disbursement": "Employee Rohan from the Finance team said his salary was delayed by 5 days this month.",
    "Incorrect Amount": "Priya reported that her March salary credited was less than expected due to a tax miscalculation.",
    "Missing Bonus": "Team lead Arjun flagged that the Diwali bonus was not credited to his account.",
    "Calculation Error": "Sneha raised a concern about incorrect bonus calculation compared to her offer letter.",
    "Balance Discrepancy": "Karan mentioned his leave balance on the portal doesn't match HR records.",
    "Approval Delay": "Meera's leave request has been pending approval from her manager for over a week.",
    "Documentation Issue": "Amit's sick leave was rejected due to missing medical certificate upload.",
    "Enrollment Issue": "Neha could not enroll in the new health insurance plan before the deadline.",
    "Claim Rejection": "Vikram's insurance claim for hospitalization was rejected without clear reason.",
    "Contribution Error": "Deepak noticed an incorrect PF contribution amount in his payslip.",
    "Verbal": "An employee reported inappropriate remarks made by a colleague during a team meeting.",
    "Discrimination": "A complaint was filed regarding biased treatment based on gender during appraisal.",
    "Equipment Request": "Tanya requested a new laptop as her current one is over 5 years old.",
    "Seating Issue": "Rahul complained about lack of proper seating arrangement in the new office floor.",
}

rows = []
for i in range(1, 61):
    category = random.choice(list(categories.keys()))
    topic = random.choice(list(categories[category].keys()))
    subtopic = random.choice(categories[category][topic])
    intent = random.choice(intents)
    sentiment = random.choice(sentiments)
    feedback = feedback_examples[subtopic]

    rows.append({
        "Ticket Number": f"TCK-{1000+i}",
        "Category": category,
        "Topic": topic,
        "Subtopic": subtopic,
        "Intent": intent,
        "Feedback Summary": feedback,
        "Sentiment": sentiment,
    })

df = pd.DataFrame(rows)
# df.to_excel("hr_tickets_sample.xlsx", index=False)
df.head()

,Ticket Number,Category,Topic,Subtopic,Intent,Feedback Summary,Sentiment
0,TCK-1001,Payroll,Salary,Incorrect Amount,Query,Priya reported that her March salary credited ...,Positive
1,TCK-1002,Leave,Annual Leave,Balance Discrepancy,Feedback,Karan mentioned his leave balance on the porta...,Positive
2,TCK-1003,Payroll,Salary,Late Disbursement,Query,Employee Rohan from the Finance team said his ...,Neutral
3,TCK-1004,Payroll,Salary,Incorrect Amount,Query,Priya reported that her March salary credited ...,Negative
4,TCK-1005,Benefits,Insurance,Enrollment Issue,Feedback,Neha could not enroll in the new health insura...,Negative


In [4]:
# pip install spacy
# python -m spacy download en_core_web_sm

import spacy

nlp = spacy.load("en_core_web_sm")

STOP_LABELS_KEEP = {"PERSON", "ORG", "GPE", "DATE", "PRODUCT"}  # entity types worth keeping

def extract_concepts(text: str):
    """
    Returns a set of normalized keyword/entity strings from a feedback sentence.
    Combines named entities (people, orgs, dates) + important noun chunks (concepts).
    """
    doc = nlp(text)
    concepts = set()

    # Named entities (e.g., 'Rohan', 'Finance team', 'March')
    for ent in doc.ents:
        if ent.label_ in STOP_LABELS_KEEP:
            concepts.add(ent.text.strip())

    # Noun chunks as generic concepts (e.g., 'salary', 'leave balance', 'medical certificate')
    for chunk in doc.noun_chunks:
        text_clean = chunk.text.lower().strip()
        # filter out pronouns / very short / stopword-only chunks
        if len(text_clean) > 2 and not chunk.root.is_stop:
            concepts.add(text_clean)

    return concepts

# quick test
extract_concepts(df.loc[0, "Feedback Summary"])

{'March', 'a tax miscalculation', 'her march salary', 'priya'}

In [5]:
# !python -m spacy download en_core_web_sm

import networkx as nx

def build_combined_graph(df: pd.DataFrame) -> nx.DiGraph:
    G = nx.DiGraph()

    for _, row in df.iterrows():
        ticket = f"Ticket:{row['Ticket Number']}"
        category = f"Category:{row['Category']}"
        topic = f"Topic:{row['Topic']}"
        subtopic = f"Subtopic:{row['Subtopic']}"
        intent = f"Intent:{row['Intent']}"
        sentiment = f"Sentiment:{row['Sentiment']}"

        # structural nodes
        G.add_node(ticket, type="Ticket")
        G.add_node(category, type="Category")
        G.add_node(topic, type="Topic")
        G.add_node(subtopic, type="Subtopic")
        G.add_node(intent, type="Intent")
        G.add_node(sentiment, type="Sentiment")

        # structural edges
        G.add_edge(ticket, category, relation="HAS_CATEGORY")
        G.add_edge(category, topic, relation="HAS_TOPIC")
        G.add_edge(topic, subtopic, relation="HAS_SUBTOPIC")
        G.add_edge(ticket, intent, relation="HAS_INTENT")
        G.add_edge(ticket, sentiment, relation="HAS_SENTIMENT")

        # NLP-derived nodes from Feedback Summary
        concepts = extract_concepts(row["Feedback Summary"])
        for concept in concepts:
            keyword_node = f"Keyword:{concept}"
            G.add_node(keyword_node, type="Keyword")
            G.add_edge(ticket, keyword_node, relation="MENTIONS")

    return G

G = build_combined_graph(df)
print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

Nodes: 149, Edges: 450


In [6]:
from pyvis.network import Network

def visualize_combined(G, output_file="hr_combined_knowledge_graph.html"):
    net = Network(height="850px", width="100%", directed=True, notebook=False)
    net.barnes_hut(gravity=-3000, spring_length=120)

    color_map = {
        "Ticket": "#8ecae6",
        "Category": "#ffb703",
        "Topic": "#fb8500",
        "Subtopic": "#e76f51",
        "Intent": "#06d6a0",
        "Sentiment": "#ef476f",
        "Keyword": "#9d4edd",   # NLP-derived nodes stand out in purple
    }
    size_map = {"Keyword": 10, "Ticket": 15}

    for node, data in G.nodes(data=True):
        label = node.split(":", 1)[1]
        net.add_node(
            node, label=label,
            color=color_map.get(data["type"], "gray"),
            size=size_map.get(data["type"], 20),
            title=data["type"],
        )
    for u, v, data in G.edges(data=True):
        net.add_edge(u, v, title=data["relation"])

    net.show(output_file, notebook=False)

visualize_combined(G)

hr_combined_knowledge_graph.html


In [7]:
Problem Will be node, Subnode Nodes containing problems, We will be containing ,  Root cause Analysis

SyntaxError: invalid syntax (3592290641.py, line 1)

In [ ]:
1. What are the major promblems with respected compensation,
2. Biased treamtment , Who are the team members who faced baised
3. Who are the people complaining about seating 

In [ ]:
Helpdesk, Why ticket raised, Summary,  I am not getting my salary in time. What Will be 

Salary will be node.  Not created is Edge,  

SyntaxError: invalid syntax (1882815756.py, line 1)

In [1]:
import pandas as pd

def generate_sample_data():
    rows = [
        # ---- Compensation / Salary ----
        ("T001","Compensation","Salary","Salary Calculation","Complaint",
         "Salary slip was not created this month due to missing calculation of overtime hours.","Negative"),
        ("T002","Compensation","Salary","Salary Calculation","Complaint",
         "Vikram reported that his salary was not created properly, missing calculation of his night shift allowance.","Negative"),
        ("T003","Compensation","Salary","Salary Disbursement","Complaint",
         "Salary credited three days late this month, causing EMI bounce.","Negative"),
        ("T004","Compensation","Bonus","Bonus Eligibility","Query",
         "Employee asked why the annual bonus was not credited despite meeting targets.","Negative"),
        ("T005","Compensation","Salary","Salary Calculation","Complaint",
         "Missing calculation in the salary slip again, tax deduction shown incorrectly.","Negative"),
        ("T006","Compensation","Salary","Salary Hike","Query",
         "Employee wants clarification on the salary hike percentage applied this appraisal cycle.","Neutral"),
        ("T007","Compensation","Salary","Salary Disbursement","Complaint",
         "Salary not created for contract employees this cycle, payroll system error.","Negative"),

        # ---- Behavior / Bias ----
        ("T008","Behavior","Bias","Biased Treatment","Complaint",
         "Rohan feels his manager showed biased treatment while assigning shifts, favoring certain teammates.","Negative"),
        ("T009","Behavior","Bias","Biased Treatment","Complaint",
         "Priya complained that biased treatment during the appraisal cycle led to an unfair rating.","Negative"),
        ("T010","Behavior","Bias","Promotion Bias","Complaint",
         "Karan alleges biased treatment in the promotion round, claims less experienced peers were promoted instead.","Negative"),
        ("T011","Behavior","Harassment","Verbal Harassment","Complaint",
         "Sneha reported verbal harassment from a team lead during a project review call.","Negative"),
        ("T012","Behavior","Bias","Biased Treatment","Complaint",
         "Arjun says biased treatment by his supervisor resulted in him being denied a preferred project.","Negative"),

        # ---- Facilities / Seating ----
        ("T013","Facilities","Seating","Seating Arrangement","Complaint",
         "Ananya complained about her seating being too close to the pantry causing noise disturbance.","Negative"),
        ("T014","Facilities","Seating","Seating Arrangement","Complaint",
         "Devika raised a seating complaint stating her desk has no proper lighting.","Negative"),
        ("T015","Facilities","Seating","Seating Allocation","Complaint",
         "Manoj complained about seating allocation, says he was moved without prior notice.","Negative"),
        ("T016","Facilities","Parking","Parking Availability","Complaint",
         "Employees reported insufficient parking slots during peak hours.","Negative"),
        ("T017","Facilities","Cafeteria","Food Quality","Complaint",
         "Feedback on cafeteria food quality has declined, employees want better hygiene.","Negative"),

        # ---- Leave / Policy ----
        ("T018","Leave","Leave Approval","Approval Delay","Complaint",
         "Leave approval delayed by over a week, causing travel plan disruption.","Negative"),
        ("T019","Leave","Leave Policy","Policy Clarity","Query",
         "Employee wants clarity on carry-forward rules for unused sick leave.","Neutral"),
        ("T020","Policy","WFH Policy","Policy Clarity","Query",
         "Query regarding the updated hybrid work policy and mandatory office days.","Neutral"),

        # ---- IT ----
        ("T021","IT Support","Access","System Access","Complaint",
         "Employee unable to access payroll portal for over a week, access request pending.","Negative"),
        ("T022","IT Support","Hardware","Laptop Issue","Complaint",
         "Laptop replacement request pending for three weeks, current laptop overheating.","Negative"),

        # ---- Positive samples ----
        ("T023","Compensation","Salary","Salary Disbursement","Feedback",
         "Salary was credited on time this month, no issues.","Positive"),
        ("T024","Facilities","Seating","Seating Arrangement","Feedback",
         "New seating arrangement near the window is appreciated by the team.","Positive"),
        ("T025","Behavior","Recognition","Peer Recognition","Feedback",
         "Team lead publicly recognized good performance, felt motivating.","Positive"),
    ]
    cols = ["Ticket Number","Category","Topic","Subtopic","Intent","Feedback Summary","Sentiments"]
    return pd.DataFrame(rows, columns=cols)

df = generate_sample_data()
df.head()

,Ticket Number,Category,Topic,Subtopic,Intent,Feedback Summary,Sentiments
0,T001,Compensation,Salary,Salary Calculation,Complaint,Salary slip was not created this month due to ...,Negative
1,T002,Compensation,Salary,Salary Calculation,Complaint,Vikram reported that his salary was not create...,Negative
2,T003,Compensation,Salary,Salary Disbursement,Complaint,"Salary credited three days late this month, ca...",Negative
3,T004,Compensation,Bonus,Bonus Eligibility,Query,Employee asked why the annual bonus was not cr...,Negative
4,T005,Compensation,Salary,Salary Calculation,Complaint,"Missing calculation in the salary slip again, ...",Negative


In [4]:
from keybert import KeyBERT

kw_model = KeyBERT(model="all-MiniLM-L6-v2")

def extract_issue_keybert(feedback_text, top_n=1):
    keywords = kw_model.extract_keywords(
        feedback_text,
        keyphrase_ngram_range=(2, 4),
        stop_words="english",
        top_n=top_n,
    )
    return keywords[0][0].title() if keywords else feedback_text[:40].title()

c:\Users\Shravan\miniforge3\envs\LLM_chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2323.98it/s]


In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def canonicalize_issues(issue_phrases, distance_threshold=0.35):
    """
    issue_phrases: list of raw extracted phrases (duplicates expected across tickets)
    returns: dict mapping raw_phrase -> canonical_phrase
    """
    unique_phrases = list(set(issue_phrases))
    if len(unique_phrases) <= 1:
        return {p: p for p in unique_phrases}

    embeddings = embedder.encode(unique_phrases, normalize_embeddings=True)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=distance_threshold,   # lower = stricter merging
        metric="cosine",
        linkage="average",
    ).fit(embeddings)

    clusters = {}
    for phrase, label in zip(unique_phrases, clustering.labels_):
        clusters.setdefault(label, []).append(phrase)

    mapping = {}
    for phrases in clusters.values():
        canonical = min(phrases, key=len)   # shortest phrase in the cluster as the label
        for p in phrases:
            mapping[p] = canonical
    return mapping

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7260.97it/s]


In [8]:
import networkx as nx
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_people(feedback_text):
    doc = nlp(feedback_text)
    return [ent.text for ent in doc.ents if ent.label_ == "PERSON"]

def extract_all_issues(df, extractor="keybert", model="llama3"):
    # dedupe identical feedback text so we don't repeat extraction work
    unique_feedback = df["Feedback Summary"].drop_duplicates()
    row_lookup = df.drop_duplicates(subset="Feedback Summary").set_index("Feedback Summary")

    cache = {}
    for feedback in unique_feedback:
        row = row_lookup.loc[feedback]
        if extractor == "llm":
            cache[feedback] = extract_issue_llm(row["Category"], row["Topic"], row["Subtopic"], feedback, model=model)
        else:
            cache[feedback] = extract_issue_keybert(feedback)
    return cache

def build_kg_dynamic(df, extractor="keybrt", model="llama3"):
    G = nx.MultiDiGraph()

    def add_node(name, ntype):
        if not G.has_node(name):
            G.add_node(name, type=ntype)

    print("Extracting root-cause issues...")
    feedback_to_issue = extract_all_issues(df, extractor=extractor, model=model)

    print("Canonicalizing similar issues...")
    canonical_map = canonicalize_issues(list(feedback_to_issue.values()))

    for _, row in df.iterrows():
        ticket = f"Ticket:{row['Ticket Number']}"
        category, topic, subtopic = row["Category"], row["Topic"], row["Subtopic"]
        intent, sentiment = row["Intent"], row["Sentiments"]
        feedback = row["Feedback Summary"]
        sentiment_node = f"{sentiment} Feedback"

        add_node(category, "Category"); add_node(topic, "Topic")
        add_node(subtopic, "Subtopic"); add_node(intent, "Intent")
        add_node(ticket, "Ticket"); add_node(sentiment_node, "Sentiment")

        G.add_edge(category, topic, label="HAS_TOPIC")
        G.add_edge(topic, subtopic, label="HAS_SUBTOPIC")
        G.add_edge(ticket, subtopic, label="REPORTED_UNDER")
        G.add_edge(ticket, intent, label="HAS_INTENT")
        G.add_edge(ticket, sentiment_node, label="HAS_SENTIMENT")

        raw_issue = feedback_to_issue[feedback]
        issue = canonical_map.get(raw_issue, raw_issue)
        add_node(issue, "RootCause")
        G.add_edge(subtopic, issue, label="HAS_ISSUE")
        G.add_edge(ticket, issue, label="REPORTS_ISSUE")
        G.add_edge(issue, sentiment_node, label="RESULTS_IN")

        for person in extract_people(feedback):
            add_node(person, "Person")
            G.add_edge(person, ticket, label="RAISED")

        G.nodes[ticket]["feedback_text"] = feedback

    return G

# use Option A (LLM) or Option B (KeyBERT) — same graph shape either way
G = build_kg_dynamic(df, extractor="keybert")     # or extractor="keybert"
print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

Extracting root-cause issues...
Canonicalizing similar issues...
89 nodes, 204 edges


In [9]:
from pyvis.network import Network

COLOR_MAP = {
    "Category":"#4C72B0","Topic":"#55A868","Subtopic":"#8172B2",
    "Intent":"#CCB974","Ticket":"#999999","RootCause":"#C44E52",
    "Sentiment":"#DD8452","Person":"#64B5CD",
}

def render_graph(graph, filename="hr_knowledge_graph.html", subset_nodes=None):
    net = Network(height="800px", width="100%", directed=True, notebook=False)
    nodes = subset_nodes if subset_nodes else graph.nodes()
    sub = graph.subgraph(nodes)
    for n, data in sub.nodes(data=True):
        net.add_node(n, label=n, color=COLOR_MAP.get(data.get("type"), "#CCCCCC"),
                      title=data.get("type",""))
    for u, v, data in sub.edges(data=True):
        net.add_edge(u, v, label=data.get("label",""), arrows="to")
    net.show(filename, notebook=False)
    print(f"Graph written to {filename} — open it in a browser.")

render_graph(G)

hr_knowledge_graph.html
Graph written to hr_knowledge_graph.html — open it in a browser.


In [10]:
import ollama
import re

def find_matching_nodes(graph, question):
    q = question.lower()
    matches = []
    for n, data in graph.nodes(data=True):
        name = n.lower()
        # strip "Ticket:" prefix etc for matching
        clean = re.sub(r"^ticket:", "", name)
        if clean in q or any(word in q for word in clean.split() if len(word) > 3):
            matches.append(n)
    return matches

def get_context_subgraph(graph, matched_nodes, radius=2):
    nodes = set()
    for n in matched_nodes:
        nodes |= set(nx.ego_graph(graph, n, radius=radius, undirected=True).nodes())
    return graph.subgraph(nodes)

def subgraph_to_facts(sub):
    facts = []
    for u, v, data in sub.edges(data=True):
        facts.append(f"{u} --{data.get('label')}--> {v}")
    return facts

def ask_hr_bot(question, graph, model="mistral:7b"):
    matched = find_matching_nodes(graph, question)
    if not matched:
        return "I couldn't find related nodes in the graph for that question.", None

    sub = get_context_subgraph(graph, matched)
    facts = subgraph_to_facts(sub)
    context = "\n".join(facts[:80])  # cap context size

    prompt = f"""You are an HR helpdesk root-cause analysis assistant.
Answer ONLY using the knowledge graph facts below. If the answer isn't
supported by the facts, say so. Be concise and specific (name issues,
people, categories mentioned in the facts).

Knowledge graph facts:
{context}

Question: {question}
Answer:"""

    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    answer = response["message"]["content"]
    return answer, sub

def chat_turn(question, graph):
    answer, sub = ask_hr_bot(question, graph)
    print("Q:", question)
    print("A:", answer)
    if sub is not None:
        print("\nRelated nodes & edges:")
        for u, v, data in sub.edges(data=True):
            print(f"  {u} --[{data.get('label')}]--> {v}")
        render_graph(graph, filename="answer_subgraph.html", subset_nodes=sub.nodes())
    print("-"*60)

In [14]:
chat_turn("What is Problem with Hardware and give its details", G)

Q: What is Problem with Hardware and give its details
A:  The problem with Hardware is a Laptop Issue, specifically a Laptop Replacement Request Pending. This issue was reported under Ticket: T022. The full details are that there is a pending laptop replacement request for an unspecified laptop.

Related nodes & edges:
  Laptop Issue --[HAS_ISSUE]--> Laptop Replacement Request Pending
  IT Support --[HAS_TOPIC]--> Access
  IT Support --[HAS_TOPIC]--> Hardware
  Ticket:T022 --[REPORTED_UNDER]--> Laptop Issue
  Ticket:T022 --[REPORTS_ISSUE]--> Laptop Replacement Request Pending
  Hardware --[HAS_SUBTOPIC]--> Laptop Issue
answer_subgraph.html
Graph written to answer_subgraph.html — open it in a browser.
------------------------------------------------------------


In [12]:
import json

COLOR_MAP = {
    "Category": "#4C72B0", "Topic": "#55A868", "Subtopic": "#8172B2",
    "Intent": "#CCB974", "Ticket": "#999999", "RootCause": "#C44E52",
    "Sentiment": "#DD8452", "Person": "#64B5CD",
}
DIM_COLOR = "#e0e0e0"

def render_graph_with_search(graph, filename="hr_knowledge_graph.html", subset_nodes=None):
    sub = graph.subgraph(subset_nodes) if subset_nodes else graph

    # --- build node payload ---
    vis_nodes = []
    for n, data in sub.nodes(data=True):
        ntype = data.get("type", "Other")
        vis_nodes.append({
            "id": n,
            "label": n,
            "type": ntype,
            "color": COLOR_MAP.get(ntype, "#CCCCCC"),
            "title": f"{ntype}: {n}",
        })

    # --- build edge payload (need stable ids for search/focus) ---
    vis_edges = []
    for i, (u, v, data) in enumerate(sub.edges(data=True)):
        vis_edges.append({
            "id": f"e{i}",
            "from": u,
            "to": v,
            "label": data.get("label", ""),
            "arrows": "to",
            "color": {"color": "#848484"},
            "font": {"align": "middle", "size": 10},
        })

    nodes_json = json.dumps(vis_nodes)
    edges_json = json.dumps(vis_edges)

    html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<script src="https://unpkg.com/vis-network@9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
  body {{ font-family: Arial, sans-serif; margin: 0; }}
  #search-bar {{
    padding: 10px; background: #222; display: flex; gap: 8px; align-items: center;
  }}
  #search-bar input {{
    flex: 1; padding: 8px; font-size: 14px; border-radius: 4px; border: none;
  }}
  #search-bar button {{
    padding: 8px 14px; border: none; border-radius: 4px; background: #555; color: white; cursor: pointer;
  }}
  #match-count {{ color: #ddd; font-size: 13px; min-width: 140px; }}
  #results-panel {{
    max-height: 160px; overflow-y: auto; background: #fafafa; border-bottom: 1px solid #ccc;
    display: none; padding: 6px 10px; font-size: 13px;
  }}
  #results-panel div {{ padding: 3px 0; cursor: pointer; }}
  #results-panel div:hover {{ text-decoration: underline; color: #2b6cb0; }}
  #mynetwork {{ width: 100%; height: 780px; border-top: 1px solid #ccc; }}
</style>
</head>
<body>

<div id="search-bar">
  <input id="search-input" type="text" placeholder="Search nodes or edges (e.g. 'Salary', 'Biased', 'HAS_ISSUE', 'Negative')" />
  <button onclick="clearSearch()">Clear</button>
  <span id="match-count"></span>
</div>
<div id="results-panel"></div>
<div id="mynetwork"></div>

<script>
  const rawNodes = {nodes_json};
  const rawEdges = {edges_json};

  const originalNodeColor = {{}};
  const originalEdgeColor = {{}};
  rawNodes.forEach(n => originalNodeColor[n.id] = n.color);
  rawEdges.forEach(e => originalEdgeColor[e.id] = e.color.color);

  const nodesDataSet = new vis.DataSet(rawNodes);
  const edgesDataSet = new vis.DataSet(rawEdges);

  const container = document.getElementById("mynetwork");
  const data = {{ nodes: nodesDataSet, edges: edgesDataSet }};
  const options = {{
    physics: {{ stabilization: true, barnesHut: {{ gravitationalConstant: -8000, springLength: 120 }} }},
    interaction: {{ hover: true, tooltipDelay: 100 }},
    edges: {{ smooth: {{ type: "dynamic" }} }},
    nodes: {{ shape: "dot", size: 16, font: {{ size: 13 }} }},
  }};
  const network = new vis.Network(container, data, options);

  const input = document.getElementById("search-input");
  const countLabel = document.getElementById("match-count");
  const resultsPanel = document.getElementById("results-panel");

  function resetHighlight() {{
    nodesDataSet.update(rawNodes.map(n => ({{
      id: n.id, color: originalNodeColor[n.id], opacity: 1,
      font: {{ color: "#000000", size: 13 }}
    }})));
    edgesDataSet.update(rawEdges.map(e => ({{
      id: e.id, color: {{ color: originalEdgeColor[e.id] }}, width: 1
    }})));
    resultsPanel.style.display = "none";
    countLabel.textContent = "";
  }}

  function clearSearch() {{
    input.value = "";
    resetHighlight();
  }}

  function runSearch() {{
    const q = input.value.trim().toLowerCase();
    if (!q) {{ resetHighlight(); return; }}

    const matchedNodeIds = new Set(
      rawNodes.filter(n => n.label.toLowerCase().includes(q) || n.type.toLowerCase().includes(q))
              .map(n => n.id)
    );
    const matchedEdges = rawEdges.filter(e => e.label.toLowerCase().includes(q));
    const matchedEdgeIds = new Set(matchedEdges.map(e => e.id));

    // also highlight nodes connected by a matched edge
    matchedEdges.forEach(e => {{ matchedNodeIds.add(e.from); matchedNodeIds.add(e.to); }});

    // dim everything, then re-highlight matches
    nodesDataSet.update(rawNodes.map(n => {{
      const isMatch = matchedNodeIds.has(n.id);
      return {{
        id: n.id,
        color: isMatch ? originalNodeColor[n.id] : DIM_COLOR_PLACEHOLDER,
        font: {{ color: isMatch ? "#000000" : "#cccccc", size: 13 }}
      }};
    }}));
    edgesDataSet.update(rawEdges.map(e => {{
      const isMatch = matchedEdgeIds.has(e.id);
      return {{
        id: e.id,
        color: {{ color: isMatch ? "#d62728" : "{DIM_COLOR}" }},
        width: isMatch ? 3 : 1
      }};
    }}));

    countLabel.textContent = matchedNodeIds.size + " node(s), " + matchedEdgeIds.size + " edge(s) found";

    // results panel: clickable list
    resultsPanel.innerHTML = "";
    if (matchedNodeIds.size > 0 || matchedEdgeIds.size > 0) {{
      resultsPanel.style.display = "block";
      [...matchedNodeIds].forEach(id => {{
        const div = document.createElement("div");
        div.textContent = "\u25CF Node: " + id;
        div.onclick = () => {{
          network.focus(id, {{ scale: 1.6, animation: true }});
          network.selectNodes([id]);
        }};
        resultsPanel.appendChild(div);
      }});
      matchedEdges.forEach(e => {{
        const div = document.createElement("div");
        div.textContent = "\u2192 Edge: " + e.from + " --[" + e.label + "]--> " + e.to;
        div.onclick = () => {{
          network.fit({{ nodes: [e.from, e.to], animation: true }});
          network.selectEdges([e.id]);
        }};
        resultsPanel.appendChild(div);
      }});
    }} else {{
      resultsPanel.style.display = "none";
    }}

    if (matchedNodeIds.size > 0) {{
      network.fit({{ nodes: [...matchedNodeIds], animation: true }});
    }}
  }}

  input.addEventListener("input", runSearch);
</script>
</body>
</html>
"""
    html = html.replace("DIM_COLOR_PLACEHOLDER", f'"{DIM_COLOR}"')

    with open(filename, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Graph with search written to {filename} — open it in a browser.")